# CISM Tutorial 01: Data Preparation

This notebook is the first step in the CISM workflow.

Its only goal is to help you finish with a dataset folder that contains:

- one `Patient_<patient_id>_FOV<fov>.txt` file per field of view
- one `patient_class.csv`

Once you have that folder, you are ready for the next stage: FANMOD+ and `CISM` initialization.

## What This Notebook Covers

CISM supports three preparation routes:

1. **Annotated centroids table** -> build spatial graph -> write colored edge-list txt files
2. **Annotated edge table** -> write colored edge-list txt files directly
3. **Prebuilt graphs** -> convert graphs to colored edge-list txt files

Run **only the section that matches your raw data**.

At the end, use the validation section to confirm that your folder is ready for `CISM.add_dataset(...)`.

## Output Contract

The target folder should look like this:

```text
data/
  my_dataset/
    Patient_1_FOV1.txt
    Patient_1_FOV2.txt
    Patient_2_FOV1.txt
    ...
    patient_class.csv
```

Each `txt` file must contain rows in this format:

```text
<src_id> <dst_id> <src_type_id> <dst_type_id>
```

`patient_class.csv` must map each patient to its class label or value.

In [ ]:
from pathlib import Path

import networkx as nx
import pandas as pd

from cism import (
    prepare_from_centroids,
    prepare_from_edge_annotations,
    prepare_from_graphs,
    validate_network_dataset_directory,
)


In [ ]:
# Edit these paths once and reuse them throughout the notebook.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "tutorials" else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "data"
DATASET_NAME = "example_dataset"
OUTPUT_DATASET_DIR = DATA_ROOT / DATASET_NAME

OUTPUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Prepared dataset output directory: {OUTPUT_DATASET_DIR}")

# These are the exact handoff values you will need in tutorial 02.
network_dataset_root_path = str(DATA_ROOT)
dataset_folder = DATASET_NAME

print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")


In [ ]:
def print_next_step_handoff(prep_result=None):
    print("Use these values in tutorial 02:")
    print(f"network_dataset_root_path = {network_dataset_root_path}")
    print(f"dataset_folder = {dataset_folder}")
    print(f"dataset directory on disk = {OUTPUT_DATASET_DIR}")
    if prep_result is not None:
        print(f"number of graph txt files written = {len(prep_result.files)}")
        print(f"preparation route = {prep_result.route}")


## Shared Notes Before You Start

- `patient_id` and `fov` together define one output graph file.
- The preparation functions normalize column names on a copy, so your raw dataframe does not need to already use CISM's canonical names.
- If your cell-type labels are already exactly what you want, use `cell_type_mapper=None`.
- If you want to merge or rename cell types, pass a mapping dictionary.
- The output of every route is a folder of `txt` graph files, not a `CISM` object yet.

## Route 1: Start From Annotated Centroids

Use this section if your raw data is a table of cells with:

- patient id
- FOV id
- x/y centroid coordinates
- cell type

This route builds a spatial graph and then writes one `txt` file per FOV.

In [ ]:
# Example: replace with your own centroid file.
# raw_centroids = pd.read_csv("path/to/your_centroids.csv")

# Example column mapping from raw names to the canonical preparation schema.
centroid_column_mapping = {
    "patient_id": "patient",
    "fov": "fov",
    "cell_type": "pred",
    "centroid-0": "centroid_x",
    "centroid-1": "centroid_y",
}

# Optional: merge fine-grained labels into shared labels.
cell_type_mapper = None
# Example:
# cell_type_mapper = {
#     "Tumor-1": "Tumor",
#     "Tumor-2": "Tumor",
#     "CD8 T": "TCell",
# }

# Uncomment once raw_centroids is loaded.
# prep_result = prepare_from_centroids(
#     dataframe=raw_centroids,
#     path_to_output_dir=str(OUTPUT_DATASET_DIR),
#     column_mapping=centroid_column_mapping,
#     cell_type_mapper=cell_type_mapper,
#     max_distance=100,
#     exclude_cell_type=None,
# )
# prep_result
# print_next_step_handoff(prep_result)


## Route 2: Start From Annotated Edges

Use this section if your raw data already contains graph edges, meaning each row already describes a pair of connected cells.

Required logical fields are:

- `patient_id`
- `fov`
- `source_id`
- `target_id`
- `source_type`
- `target_type`

This route skips graph construction and writes the CISM/FANMOD input files directly.

In [ ]:
# Example: replace with your own edge-annotation file.
# raw_edges = pd.read_csv("path/to/your_edge_annotations.csv")

edge_column_mapping = {
    "patient_id": "patient",
    "fov": "fov",
    "source_id": "id1",
    "target_id": "id2",
    "source_type": "type1",
    "target_type": "type2",
}

# Uncomment once raw_edges is loaded.
# prep_result = prepare_from_edge_annotations(
#     dataframe=raw_edges,
#     path_to_output_dir=str(OUTPUT_DATASET_DIR),
#     column_mapping=edge_column_mapping,
# )
# prep_result
# print_next_step_handoff(prep_result)


## Route 3: Start From Prebuilt Graphs

Use this section if you already have one graph per patient/FOV.

Each graph must be a `networkx.Graph`, and each node must have a node attribute that stores the cell type.

By default, the preparation code expects that attribute to be called `cell_type`.

In [ ]:
# Example in-memory graph.
# Replace this with your actual graph loading logic.
example_graph = nx.Graph()
example_graph.add_node("A", cell_type="Tumor")
example_graph.add_node("B", cell_type="TCell")
example_graph.add_edge("A", "B")

graph_records = [
    {
        "patient_id": "1",
        "fov": "1",
        "graph": example_graph,
    }
]

# Uncomment to run.
# prep_result = prepare_from_graphs(
#     graphs=graph_records,
#     path_to_output_dir=str(OUTPUT_DATASET_DIR),
#     node_type_attribute="cell_type",
# )
# prep_result
# print_next_step_handoff(prep_result)


## Create `patient_class.csv`

After creating the graph `txt` files, make sure the same dataset folder also contains `patient_class.csv`.

Each row should look like:

```text
<dataset_prefix><patient_id>,<class_or_value>
```

Examples:

```text
CRC1,POSITIVE
CRC2,NEGATIVE
```

or:

```text
CRC1,2612
CRC2,3822
```

In [ ]:
# Example: create patient_class.csv from a dataframe.
# Replace with your real patient metadata.
patient_class_df = pd.DataFrame(
    {
        "patient_uid": ["CRC1", "CRC2"],
        "patient_class": ["POSITIVE", "NEGATIVE"],
    }
)

# Uncomment to write the file.
# patient_class_df.to_csv(
#     OUTPUT_DATASET_DIR / "patient_class.csv",
#     index=False,
#     header=False,
# )
# patient_class_df


## Validate The Final Dataset Folder

Run this section after:

1. one of the three preparation routes
2. writing `patient_class.csv`

If validation succeeds, you are ready for the next notebook.

In [ ]:
txt_files = validate_network_dataset_directory(OUTPUT_DATASET_DIR)
patient_class_path = OUTPUT_DATASET_DIR / "patient_class.csv"

print("Validated txt graph files:")
for file_name in txt_files[:10]:
    print(" -", file_name)
if len(txt_files) > 10:
    print(f" ... and {len(txt_files) - 10} more")

print()
print(f"patient_class.csv exists: {patient_class_path.exists()}")
print(f"Dataset folder ready: {OUTPUT_DATASET_DIR}")
print()
print_next_step_handoff()


## You Are Done When...

Before moving on, confirm all of the following:

- your dataset folder contains one `Patient_<patient_id>_FOV<fov>.txt` per FOV
- every txt row has exactly 4 values: `src_id dst_id src_type_id dst_type_id`
- the same folder contains `patient_class.csv`
- `validate_network_dataset_directory(...)` runs without errors

At that point, your output folder is ready for the next pipeline stage:

```python
cism.add_dataset(dataset_folder=..., dataset_type=..., dataset_name=...)
```

Recommended next notebook: **FANMOD+ and CISM initialization**.